# Case study · Five questions about the Berlin Airbnb listings

*Session 1, block 2 practice. Author: course team, licence CC-BY-4.0.*

**Why this matters.** Berlin restricts holiday rentals: since 2014 a flat may only be let to tourists with a permit, and since 2018 every listing needs a registration number. Housing analysts in the districts, journalists and Airbnb's own policy team all start with the same simple questions: how many listings are there, where are they, what kind of accommodation is offered, what does it cost, and how much is it used? You answer these five questions with pandas for the 12,776 Berlin listings of the Inside Airbnb snapshot of 26 June 2026. Write each answer as **one sentence** in the Markdown cell below the code that produces it.

**Before you start**, generate the data once from the repository root (downloads about 100 MB):

```bash
uv run python case-study/prepare_airbnb.py
```

Then start JupyterLab from the repository root, for example with `uv run jupyter lab`, and open this notebook.

The questions:

1. **How many?** How many listings and how many hosts are there, and how many listings belong to hosts with more than one?
2. **Where?** How many listings are there in each of the 12 districts?
3. **What type?** What share of the listings are entire homes, private rooms, shared rooms and hotel rooms?
4. **What price?** What does a night cost by room type and by district, and why do we look only at short stays?
5. **How many reviews?** How many reviews does a typical listing have, and what share has none?

Questions 1 and 2 are worked examples. Questions 3 to 5 have hints; a reference solution is at the end. Try first.

> The data were collected from public Airbnb pages by Inside Airbnb (CC BY 4.0); the preparation script removed host names, texts and photos. Report results in aggregate and never try to identify a host.

## Setup: find the data and load it

In [ ]:
from pathlib import Path

import pandas as pd

# The notebook may be started from the repository root or from its own folder.
# Walk up the folders until we find case-study/data/airbnb/.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "case-study" / "data" / "airbnb").exists())
DATA = root / "case-study" / "data" / "airbnb"

listings = pd.read_parquet(DATA / "listings.parquet")
listings.head(3)

In [ ]:
# check the types first: prices and coordinates are numbers, review dates are timestamps
listings.dtypes.head(25)

One row is one listing. The columns you need today:

In [ ]:
listings.loc[0, ["id", "name", "host_id", "district", "room_type", "accommodates",
                 "price", "minimum_nights", "number_of_reviews"]].to_dict()

## Question 1 · How many listings and hosts? (worked example)

In [ ]:
n_listings, n_columns = listings.shape
n_hosts = listings["host_id"].nunique()
print(f"{n_listings:,} listings in {n_columns} columns, offered by {n_hosts:,} hosts")

listings_per_host = listings.groupby("host_id").size()     # one number per host
print("hosts with a single listing:", round((listings_per_host == 1).mean(), 3))
print("largest number of listings of one host:", listings_per_host.max())

# share of *listings* (not hosts) that belong to a host with more than one listing
host_size = listings["host_id"].map(listings_per_host)
round((host_size > 1).mean(), 3)

*Answer:* The snapshot has 12,776 listings offered by 8,182 hosts. 84 % of the hosts offer a single listing, but the 16 % with several listings hold 46 % of all listings; one host offers 261.

> **Why it matters.** "Airbnb in Berlin" means two different things: people who let their own flat now and then, and professional operators with many flats. Questions about housing policy depend on which group you count.

## Question 2 · Listings per district (worked example)

In [ ]:
per_district = listings["district"].value_counts()
print(listings["district"].nunique(), "districts")
pd.DataFrame({"listings": per_district,
              "share": (per_district / len(listings)).round(3)})

*Answer:* More than two thirds of the listings are in four inner-city districts: Mitte (2,826, 22 %), Friedrichshain-Kreuzberg (21 %), Pankow (15 %) and Charlottenburg-Wilmersdorf (11 %). The outer districts Spandau and Marzahn-Hellersdorf have fewer than 140 listings each.

> **Why it matters.** Counts depend on district size. Session 3 joins the listings with other tables, Session 5 compares districts with statistical tests.

## Question 3 · Room types

*Hints:* `value_counts(normalize=True)` on `room_type` gives the shares. There are exactly four room types; which two cover almost all listings?

In [ ]:
# your code for question 3

*Answer:* ...

## Question 4 · What does a night cost?

*Hints:* `price` is the price per night in euros; it is missing for about a third of the listings. Look first at `listings["minimum_nights"].value_counts().head()`: many listings can only be booked for 92 nights or more. Such monthly rentals report a price per night that is not comparable with a weekend stay, so the course compares prices only for **short stays**: a price present and `minimum_nights < 28`. Build this subset with a boolean mask, then group its `price` by `room_type` and by `district` and take the `median()`. Why the median and not the mean? Look at the most expensive listing.

In [ ]:
# your code for question 4

*Answer:* ...

## Question 5 · Reviews

*Hints:* `number_of_reviews` counts all reviews since the listing was created, `number_of_reviews_ltm` those of the last twelve months. Compare the median and the mean, and compute the share of listings with no review at all (the mean of a boolean Series is the share of `True`). Reviews are the course's measure of demand: a guest can only review a stay that took place.

In [ ]:
# your code for question 5

*Answer:* ...

## Checking AI-generated code

If you used an AI coding assistant, check its code before you trust the number:

- Compute one value in a second, independent way (for example a filter and `len()` instead of `value_counts()`).
- Try an edge case: what does the code do with a listing whose price is missing?
- Record the tool, the prompt and how you checked the answer in your notes (later in `AI_USE.md` of your team repository).

In [ ]:
# Example of an independent check for question 2: count the listings in Mitte with a mask
in_mitte = listings["district"] == "Mitte"
assert in_mitte.sum() == listings["district"].value_counts()["Mitte"]
round(in_mitte.mean(), 3)

---

## Reference solution

Try the questions yourself first. The cells below compute all five answers.

In [ ]:
# Question 3
listings["room_type"].value_counts(normalize=True).round(3)

In [ ]:
# Question 4
print(listings["minimum_nights"].value_counts().head())         # 92 nights: rented by the quarter
short = listings[listings["price"].notna() & (listings["minimum_nights"] < 28)]
print(f"{len(short):,} short-stay listings with a price")
print("most expensive:", short["price"].max(), "EUR; median", short["price"].median(),
      "mean", round(short["price"].mean(), 1))
short.groupby("room_type")["price"].agg(["size", "median"])

In [ ]:
short.groupby("district")["price"].median().sort_values()

In [ ]:
# Question 5
reviews = listings["number_of_reviews"]
print("median", reviews.median(), "mean", round(reviews.mean(), 1), "max", reviews.max())
print("share without any review:", round((reviews == 0).mean(), 3))
print("share with a review in the last 12 months:", round((listings["number_of_reviews_ltm"] > 0).mean(), 3))

*Reference answers.* (3) 69 % of the listings are entire homes and 29 % private rooms; hotel rooms and shared rooms make up less than 1 % each. (4) 6,701 listings are short stays with a price; a night costs €157 in the median, €185 for an entire home and €97 for a private room. Mitte is the most expensive district (median €187), Reinickendorf the cheapest (€100). The mean (€194) lies above the median because a few listings ask for more than €1,000, one even €10,025: the price distribution is skewed, and the median is the better summary. Monthly rentals (28 nights or more, mostly 92) report much lower prices per night (median €23) and would distort the comparison. (5) The median listing has 11 reviews, the mean is 53 and the maximum 2,832; 20 % of the listings have never been reviewed, and only 55 % had a review in the last twelve months, so many listings are rarely or no longer booked.

## Next step

In block 3 the same analysis becomes a module with a class, `ListingTable`, in [`../workspace/`](../workspace/). Its methods answer these five questions, and tests check them on eight hand-made listings.